In [ ]:
import duckdb
import pandas as pd

from IPython.display import display
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
pd.get_option("display.max_rows")

import os
from pathlib import Path

con = duckdb.connect()

print(os.getcwd())
raw_data_path = Path("/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/raw")
os.listdir(raw_data_path)


# source data: https://www.kaggle.com/datasets/lzytim/hdb-resale-prices

/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/notebooks


['clean_eval.csv',
 'hdb_resale_index.csv',
 'hold.csv',
 'hdb_resale_flat_transactions.csv',
 'train.csv',
 'clean_train.csv',
 'eval.csv']

### CREATE table in duckdb:
* hdb_resale

In [2]:
raw_file_path = raw_data_path/'hdb_resale_flat_transactions.csv'
con.execute(f"CREATE TABLE hdb_resale AS SELECT * FROM '{raw_file_path}'")
con.sql("""SHOW TABLES""").show()

┌────────────┐
│    name    │
│  varchar   │
├────────────┤
│ hdb_resale │
└────────────┘



In [3]:
qry = """
SELECT * FROM hdb_resale
limit 5
"""
con.sql(qry).df()

,column00,month,town,blk_no,road_name,building,postal,resale_price,storey_range,flat_type,flat_model,lease_commence_date,remaining_lease_years,remaining_lease_months,floor_area_sqm,floor_area_sqft,price_per_sqft,planning_area_ura,region_ura,x,y,latitude,longitude,closest_mrt_station,distance_to_mrt_meters,transport_type,line_color,distance_to_cbd,closest_pri_school,distance_to_pri_school_meters
0,0,2017-01-01,ANG MO KIO,406,ANG MO KIO AVENUE 10,NIL,560406,232000.0,10 TO 12,2 ROOM,Improved,1979-05-01,61,4,44.0,473.6116,489.852867,ANG MO KIO,NORTH-EAST REGION,30288.234663,38229.067463,1.362005,103.853880,Ang Mo Kio,999.941618,MRT,Red,8615.656983,TOWNSVILLE PRIMARY SCHOOL,218.125254
1,1,2017-01-01,ANG MO KIO,108,ANG MO KIO AVENUE 4,KEBUN BARU HEIGHTS,560108,250000.0,01 TO 03,3 ROOM,New Generation,1978-08-01,60,7,67.0,721.1813,346.653470,ANG MO KIO,NORTH-EAST REGION,28543.458747,39220.009892,1.370966,103.838202,Ang Mo Kio,1268.958246,MRT,Red,9715.131951,ANG MO KIO PRIMARY SCHOOL,241.572335
2,2,2017-01-01,ANG MO KIO,602,ANG MO KIO AVENUE 5,YIO CHU KANG GREEN,560602,262000.0,01 TO 03,3 ROOM,New Generation,1980-06-01,62,5,67.0,721.1813,363.292836,ANG MO KIO,NORTH-EAST REGION,28228.099954,40297.283149,1.380709,103.835368,Yio Chu Kang,1072.295164,MRT,Red,10828.819556,ANDERSON PRIMARY SCHOOL,777.155378
3,3,2017-01-01,ANG MO KIO,465,ANG MO KIO AVENUE 10,TECK GHEE HORIZON,560465,265000.0,04 TO 06,3 ROOM,New Generation,1980-02-01,62,1,68.0,731.9452,362.048962,ANG MO KIO,NORTH-EAST REGION,30657.824693,38693.098657,1.366201,103.857201,Ang Mo Kio,945.371842,MRT,Red,9097.929095,TECK GHEE PRIMARY SCHOOL,698.165530
4,4,2017-01-01,ANG MO KIO,601,ANG MO KIO AVENUE 5,YIO CHU KANG GREEN,560601,265000.0,01 TO 03,3 ROOM,New Generation,1980-06-01,62,5,67.0,721.1813,367.452678,ANG MO KIO,NORTH-EAST REGION,28201.782245,40334.052030,1.381041,103.835132,Yio Chu Kang,1095.197729,MRT,Red,10869.453109,ANDERSON PRIMARY SCHOOL,782.553222


### REMOVE csv index column

In [4]:
qry = """
ALTER TABLE hdb_resale
DROP column00
"""
con.sql(qry)

In [5]:
qry = """
SELECT DISTINCT 
EXTRACT(YEAR FROM month) AS yr
FROM hdb_resale
ORDER BY yr DESC
"""
con.sql(qry).show()

┌───────┐
│  yr   │
│ int64 │
├───────┤
│  2025 │
│  2024 │
│  2023 │
│  2022 │
│  2021 │
│  2020 │
│  2019 │
│  2018 │
│  2017 │
└───────┘



### SPLIT DATA: based on temporal change

In [6]:
qry = """
SELECT * FROM hdb_resale
WHERE EXTRACT(YEAR FROM month) = 2025
"""
hold_df = con.sql(qry).df()
hold_df

,month,town,blk_no,road_name,building,postal,resale_price,storey_range,flat_type,flat_model,lease_commence_date,remaining_lease_years,remaining_lease_months,floor_area_sqm,floor_area_sqft,price_per_sqft,planning_area_ura,region_ura,x,y,latitude,longitude,closest_mrt_station,distance_to_mrt_meters,transport_type,line_color,distance_to_cbd,closest_pri_school,distance_to_pri_school_meters
0,2025-05-01,ANG MO KIO,406,ANG MO KIO AVENUE 10,NIL,560406,340000.0,04 TO 06,2 ROOM,Improved,1979-05-01,53,1,44.0,473.6116,717.887822,ANG MO KIO,NORTH-EAST REGION,30288.234663,38229.067463,1.362005,103.853880,Ang Mo Kio,999.941618,MRT,Red,8615.656983,TOWNSVILLE PRIMARY SCHOOL,218.125254
1,2025-08-01,ANG MO KIO,406,ANG MO KIO AVENUE 10,NIL,560406,353000.0,07 TO 09,2 ROOM,Improved,1979-05-01,52,10,44.0,473.6116,745.336474,ANG MO KIO,NORTH-EAST REGION,30288.234663,38229.067463,1.362005,103.853880,Ang Mo Kio,999.941618,MRT,Red,8615.656983,TOWNSVILLE PRIMARY SCHOOL,218.125254
2,2025-01-01,ANG MO KIO,314,ANG MO KIO AVENUE 3,TECK GHEE EVERGREEN,560314,318000.0,10 TO 12,2 ROOM,Improved,1978-01-01,52,2,44.0,473.6116,671.436257,ANG MO KIO,NORTH-EAST REGION,29865.998046,38695.970271,1.366227,103.850086,Ang Mo Kio,413.954069,MRT,Red,9079.649947,TECK GHEE PRIMARY SCHOOL,119.991487
3,2025-03-01,ANG MO KIO,314,ANG MO KIO AVENUE 3,TECK GHEE EVERGREEN,560314,338000.0,10 TO 12,2 ROOM,Improved,1978-01-01,51,11,44.0,473.6116,713.664952,ANG MO KIO,NORTH-EAST REGION,29865.998046,38695.970271,1.366227,103.850086,Ang Mo Kio,413.954069,MRT,Red,9079.649947,TECK GHEE PRIMARY SCHOOL,119.991487
4,2025-03-01,ANG MO KIO,323,ANG MO KIO AVENUE 3,ANG MO KIO 31,560323,300000.0,07 TO 09,2 ROOM,Improved,1977-06-01,51,4,44.0,473.6116,633.430431,ANG MO KIO,NORTH-EAST REGION,29602.047153,38881.891694,1.367908,103.847714,Ang Mo Kio,303.675903,MRT,Red,9273.665277,TECK GHEE PRIMARY SCHOOL,442.837244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20014,2025-07-01,YISHUN,824,YISHUN STREET 81,NIL,760824,980000.0,04 TO 06,EXECUTIVE,Maisonette,1987-11-01,61,5,146.0,1571.5294,623.596351,YISHUN,NORTH REGION,27998.288199,43950.290932,1.413745,103.833303,Khatib,403.882660,MRT,Red,14474.449019,PEIYING PRIMARY SCHOOL,501.914196
20015,2025-09-01,YISHUN,877,YISHUN STREET 81,NIL,760877,980000.0,10 TO 12,EXECUTIVE,Maisonette,1987-12-01,61,3,145.0,1560.7655,627.897016,YISHUN,NORTH REGION,28237.634702,43967.636908,1.413902,103.835454,Khatib,473.260140,MRT,Red,14460.190304,NAVAL BASE PRIMARY SCHOOL,457.007677
20016,2025-09-01,YISHUN,834,YISHUN STREET 81,NIL,760834,990000.0,04 TO 06,EXECUTIVE,Maisonette,1988-01-01,61,4,146.0,1571.5294,629.959580,YISHUN,NORTH REGION,28075.777355,44098.765290,1.415088,103.834000,Khatib,278.011616,MRT,Red,14610.975312,PEIYING PRIMARY SCHOOL,463.909479
20017,2025-05-01,YISHUN,632,YISHUN STREET 61,NIL,760632,945000.0,04 TO 06,MULTI-GENERATION,Multi Generation,1987-10-01,61,6,147.0,1582.2933,597.234407,YISHUN,NORTH REGION,28653.063837,44483.489312,1.418567,103.839187,Khatib,703.050548,MRT,Red,14928.515187,NAVAL BASE PRIMARY SCHOOL,253.775129


In [7]:
qry = """
SELECT * FROM hdb_resale
WHERE EXTRACT(YEAR FROM month) IN (2024)
"""
eval_df = con.sql(qry).df()
eval_df

,month,town,blk_no,road_name,building,postal,resale_price,storey_range,flat_type,flat_model,lease_commence_date,remaining_lease_years,remaining_lease_months,floor_area_sqm,floor_area_sqft,price_per_sqft,planning_area_ura,region_ura,x,y,latitude,longitude,closest_mrt_station,distance_to_mrt_meters,transport_type,line_color,distance_to_cbd,closest_pri_school,distance_to_pri_school_meters
0,2024-02-01,ANG MO KIO,406,ANG MO KIO AVENUE 10,NIL,560406,285000.0,01 TO 03,2 ROOM,Improved,1979-05-01,54,4,44.0,473.6116,601.758910,ANG MO KIO,NORTH-EAST REGION,30288.234663,38229.067463,1.362005,103.853880,Ang Mo Kio,999.941618,MRT,Red,8615.656983,TOWNSVILLE PRIMARY SCHOOL,218.125254
1,2024-02-01,ANG MO KIO,323,ANG MO KIO AVENUE 3,ANG MO KIO 31,560323,293000.0,04 TO 06,2 ROOM,Improved,1977-06-01,52,5,44.0,473.6116,618.650388,ANG MO KIO,NORTH-EAST REGION,29602.047153,38881.891694,1.367908,103.847714,Ang Mo Kio,303.675903,MRT,Red,9273.665277,TECK GHEE PRIMARY SCHOOL,442.837244
2,2024-02-01,ANG MO KIO,314,ANG MO KIO AVENUE 3,TECK GHEE EVERGREEN,560314,303000.0,01 TO 03,2 ROOM,Improved,1978-01-01,52,11,44.0,473.6116,639.764735,ANG MO KIO,NORTH-EAST REGION,29865.998046,38695.970271,1.366227,103.850086,Ang Mo Kio,413.954069,MRT,Red,9079.649947,TECK GHEE PRIMARY SCHOOL,119.991487
3,2024-04-01,ANG MO KIO,314,ANG MO KIO AVENUE 3,TECK GHEE EVERGREEN,560314,288000.0,01 TO 03,2 ROOM,Improved,1978-01-01,52,10,44.0,473.6116,608.093214,ANG MO KIO,NORTH-EAST REGION,29865.998046,38695.970271,1.366227,103.850086,Ang Mo Kio,413.954069,MRT,Red,9079.649947,TECK GHEE PRIMARY SCHOOL,119.991487
4,2024-06-01,ANG MO KIO,323,ANG MO KIO AVENUE 3,ANG MO KIO 31,560323,300000.0,10 TO 12,2 ROOM,Improved,1977-06-01,52,1,44.0,473.6116,633.430431,ANG MO KIO,NORTH-EAST REGION,29602.047153,38881.891694,1.367908,103.847714,Ang Mo Kio,303.675903,MRT,Red,9273.665277,TECK GHEE PRIMARY SCHOOL,442.837244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27827,2024-12-01,YISHUN,834,YISHUN STREET 81,NIL,760834,950000.0,07 TO 09,EXECUTIVE,Apartment,1988-01-01,62,3,142.0,1528.4738,621.535024,YISHUN,NORTH REGION,28075.777355,44098.765290,1.415088,103.834000,Khatib,278.011616,MRT,Red,14610.975312,PEIYING PRIMARY SCHOOL,463.909479
27828,2024-12-01,YISHUN,828,YISHUN STREET 81,NIL,760828,990000.0,07 TO 09,EXECUTIVE,Maisonette,1988-01-01,62,3,146.0,1571.5294,629.959580,YISHUN,NORTH REGION,27953.606541,44110.138999,1.415191,103.832902,Khatib,242.585206,MRT,Red,14638.986228,PEIYING PRIMARY SCHOOL,357.013909
27829,2024-02-01,YISHUN,666,YISHUN AVENUE 4,NIL,760666,998000.0,04 TO 06,MULTI-GENERATION,Multi Generation,1987-12-01,62,11,164.0,1765.2796,565.349534,YISHUN,NORTH REGION,28806.756365,44531.159086,1.418998,103.840568,Khatib,863.123975,MRT,Red,14962.748464,NORTHLAND PRIMARY SCHOOL,221.649812
27830,2024-03-01,YISHUN,666,YISHUN AVENUE 4,NIL,760666,1200000.0,10 TO 12,MULTI-GENERATION,Multi Generation,1987-12-01,62,9,164.0,1765.2796,679.778999,YISHUN,NORTH REGION,28806.756365,44531.159086,1.418998,103.840568,Khatib,863.123975,MRT,Red,14962.748464,NORTHLAND PRIMARY SCHOOL,221.649812


In [8]:
qry = """
SELECT * FROM hdb_resale
WHERE EXTRACT(YEAR FROM month) < 2024
"""
train_df = con.sql(qry).df()
train_df

,month,town,blk_no,road_name,building,postal,resale_price,storey_range,flat_type,flat_model,lease_commence_date,remaining_lease_years,remaining_lease_months,floor_area_sqm,floor_area_sqft,price_per_sqft,planning_area_ura,region_ura,x,y,latitude,longitude,closest_mrt_station,distance_to_mrt_meters,transport_type,line_color,distance_to_cbd,closest_pri_school,distance_to_pri_school_meters
0,2017-01-01,ANG MO KIO,406,ANG MO KIO AVENUE 10,NIL,560406,232000.0,10 TO 12,2 ROOM,Improved,1979-05-01,61,4,44.0,473.6116,489.852867,ANG MO KIO,NORTH-EAST REGION,30288.234663,38229.067463,1.362005,103.853880,Ang Mo Kio,999.941618,MRT,Red,8615.656983,TOWNSVILLE PRIMARY SCHOOL,218.125254
1,2017-01-01,ANG MO KIO,108,ANG MO KIO AVENUE 4,KEBUN BARU HEIGHTS,560108,250000.0,01 TO 03,3 ROOM,New Generation,1978-08-01,60,7,67.0,721.1813,346.653470,ANG MO KIO,NORTH-EAST REGION,28543.458747,39220.009892,1.370966,103.838202,Ang Mo Kio,1268.958246,MRT,Red,9715.131951,ANG MO KIO PRIMARY SCHOOL,241.572335
2,2017-01-01,ANG MO KIO,602,ANG MO KIO AVENUE 5,YIO CHU KANG GREEN,560602,262000.0,01 TO 03,3 ROOM,New Generation,1980-06-01,62,5,67.0,721.1813,363.292836,ANG MO KIO,NORTH-EAST REGION,28228.099954,40297.283149,1.380709,103.835368,Yio Chu Kang,1072.295164,MRT,Red,10828.819556,ANDERSON PRIMARY SCHOOL,777.155378
3,2017-01-01,ANG MO KIO,465,ANG MO KIO AVENUE 10,TECK GHEE HORIZON,560465,265000.0,04 TO 06,3 ROOM,New Generation,1980-02-01,62,1,68.0,731.9452,362.048962,ANG MO KIO,NORTH-EAST REGION,30657.824693,38693.098657,1.366201,103.857201,Ang Mo Kio,945.371842,MRT,Red,9097.929095,TECK GHEE PRIMARY SCHOOL,698.165530
4,2017-01-01,ANG MO KIO,601,ANG MO KIO AVENUE 5,YIO CHU KANG GREEN,560601,265000.0,01 TO 03,3 ROOM,New Generation,1980-06-01,62,5,67.0,721.1813,367.452678,ANG MO KIO,NORTH-EAST REGION,28201.782245,40334.052030,1.381041,103.835132,Yio Chu Kang,1095.197729,MRT,Red,10869.453109,ANDERSON PRIMARY SCHOOL,782.553222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169145,2023-12-01,YISHUN,606,YISHUN STREET 61,NEE SOON CENTRAL MEADOWS,760606,788888.0,07 TO 09,EXECUTIVE,Apartment,1987-11-01,63,1,142.0,1528.4738,516.127918,YISHUN,NORTH REGION,28333.820088,44789.428103,1.421334,103.836318,Khatib,573.468277,MRT,Red,15265.137599,NORTHLAND PRIMARY SCHOOL,514.624052
169146,2023-12-01,YISHUN,328,YISHUN RING ROAD,YISHUN RIVERGREEN,760328,798000.0,07 TO 09,EXECUTIVE,Apartment,1988-07-01,63,7,142.0,1528.4738,522.089420,YISHUN,NORTH REGION,29083.732600,45723.276519,1.429780,103.843057,Yishun,896.825885,MRT,Red,16132.807368,HUAMIN PRIMARY SCHOOL,337.132036
169147,2023-12-01,YISHUN,643,YISHUN STREET 61,NIL,760643,838000.0,10 TO 12,EXECUTIVE,Maisonette,1987-09-01,62,10,146.0,1571.5294,533.238513,YISHUN,NORTH REGION,28458.318995,44789.537249,1.421335,103.837437,Khatib,661.034193,MRT,Red,15252.002713,NORTHLAND PRIMARY SCHOOL,390.639884
169148,2023-12-01,YISHUN,348D,YISHUN AVENUE 11,ADORA GREEN,764348,856000.0,04 TO 06,5 ROOM,DBSS,2013-10-01,88,10,112.0,1205.5568,710.045350,YISHUN,NORTH REGION,29065.931757,45407.814785,1.426927,103.842897,Yishun,921.278726,MRT,Red,15818.949672,HUAMIN PRIMARY SCHOOL,149.573414


In [9]:
train_df.to_csv(raw_data_path/'train.csv',index=None)
eval_df.to_csv(raw_data_path/'eval.csv',index=None)
hold_df.to_csv(raw_data_path/'hold.csv',index=None)